## Dates and saved feature-selection outputs
Training starts **after** `TRAIN_START_YEAR` / `TRAIN_START_MONTH` / `TRAIN_START_DAY`; validation starts at `VAL_START_DATE` and test starts at `TEST_START_DATE`. Edit these shared values in `src/ufc_betting/config.py`, then rerun from the imports.

Only the training split is used for the Lasso paths and ranks. The final cell exposes both figures, `coef_df`, and `rank_df`, and saves them to editable paths. The AI Input Features page reads the default filenames from `settings.feature_selection_dir`; if you move the exports, update that directory for the page as well.

In [3]:
import numpy as np
import pandas as pd


In [ ]:
import importlib
import ufc_betting.config as config_module
importlib.reload(config_module)
from ufc_betting.config import (
    settings, TRAIN_START_YEAR, TRAIN_START_MONTH, TRAIN_START_DAY,
    VAL_START_DATE, TEST_START_DATE,
)
from ufc_betting.Models.FeatureSelection.lasso import (
    plot_lasso_path, rank_features_by_lasso_path, plot_lasso_importance,
    save_lasso_outputs,
)
from ufc_betting.Models.LogisticRegression.train_test_builder import TrainTestBuilder


In [9]:
feats_to_test = [
    'td_defense_pct_red',
    'td_total_attempted_against_red',
    'td_total_landed_against_red',
    'td_defense_pct_blue',
    'td_total_attempted_against_blue',
    'td_total_landed_against_blue',
    'sig_str_defense_pct_red',
    'sig_str_total_attempted_against_red',
    'sig_str_total_landed_against_red',
    'sig_str_defense_pct_blue',
    'sig_str_total_attempted_against_blue',
    'sig_str_total_landed_against_blue',
    'td_accuracy_pct_red',
    'td_accuracy_pct_blue',
    'sig_str_accuracy_pct_red',
    'sig_str_accuracy_pct_blue',
    'kd_pm_red',
    'kd_total_red',
    'kd_pm_blue',
    'kd_total_blue',
    'sig_str_landed_pm_red',
    'sig_str_landed_total_red',
    'sig_str_landed_pm_blue',
    'sig_str_landed_total_blue',
    'sig_str_absorbed_pm_red',
    'sig_str_absorbed_total_red',
    'sig_str_absorbed_pm_blue',
    'sig_str_absorbed_total_blue',
    'sig_str_attempted_pm_red',
    'sig_str_attempted_total_red',
    'sig_str_attempted_pm_blue',
    'sig_str_attempted_total_blue',
    'leg_str_pm_red',
    'leg_str_total_red',
    'leg_str_pm_blue',
    'leg_str_total_blue',
    'head_str_pm_red',
    'head_str_total_red',
    'head_str_pm_blue',
    'head_str_total_blue',
    'body_str_pm_red',
    'body_str_total_red',
    'body_str_pm_blue',
    'body_str_total_blue',
    'clinch_str_pm_red',
    'clinch_str_total_red',
    'clinch_str_pm_blue',
    'clinch_str_total_blue',
    'td_landed_pm_red',
    'td_landed_total_red',
    'td_landed_pm_blue',
    'td_landed_total_blue',
    'td_attempted_pm_red',
    'td_attempted_total_red',
    'td_attempted_pm_blue',
    'td_attempted_total_blue',
    'td_defended_pm_red',
    'td_defended_total_red',
    'td_defended_pm_blue',
    'td_defended_total_blue',
    'control_pm_red',
    'control_total_red',
    'control_pm_blue',
    'control_total_blue',
    'sub_att_pm_red',
    'sub_att_total_red',
    'sub_att_pm_blue',
    'sub_att_total_blue',
    'reverse_pm_red',
    'reverse_total_red',
    'reverse_pm_blue',
    'reverse_total_blue',
    'age_red',
    'age_blue',
    'height_red',
    'height_blue',
    'reach_red',
    'reach_blue',
    'bout_position',
    'ratio_td_red',
    'ratio_td_blue',
    'ratio_control_red',
    'ratio_control_blue',
    'ratio_sig_str_red',
    'ratio_sig_str_blue',
    'adjusted_sig_str_red',
    'adjusted_sig_str_blue',
    'adjusted_control_red',
    'adjusted_control_blue',
    'adjusted_td_red',
    'adjusted_td_blue',
    'ratio_td_diff',
    'ratio_control_diff',
    'ratio_sigstrike_diff',
    'kd_total_diff',
    'kd_pm_diff',
    'sig_str_landed_total_diff',
    'sig_str_landed_pm_diff',
    'sig_str_absorbed_total_diff',
    'sig_str_absorbed_pm_diff',
    'sig_str_attempted_total_diff',
    'sig_str_attempted_pm_diff',
    'leg_str_total_diff',
    'leg_str_pm_diff',
    'head_str_total_diff',
    'head_str_pm_diff',
    'body_str_total_diff',
    'body_str_pm_diff',
    'clinch_str_total_diff',
    'clinch_str_pm_diff',
    'td_landed_total_diff',
    'td_landed_pm_diff',
    'td_attempted_total_diff',
    'td_attempted_pm_diff',
    'control_total_diff',
    'control_pm_diff',
    'sub_att_total_diff',
    'sub_att_pm_diff',
    'reverse_total_diff',
    'reverse_pm_diff',
    'age_diff',
    'height_diff',
    'reach_diff',
    'sig_str_defense_pct_diff',
    'sig_str_accuracy_pct_diff',
    'td_accuracy_pct_diff',
    'td_defense_pct_diff',
    'opp_age_red',
    'opp_age_blue',
    'opp_reach_red',
    'opp_reach_blue',
    'avg_fight_min_red',
    'avg_fight_min_blue',
    'avg_fight_min_diff',
    'total_bonus_red',
    'total_bonus_blue',
    'total_bonus_diff',
    'elo_red',
    'elo_blue',
    'elo_red_proba',
    'elo_blue_proba',
    'elo_diff',
    'elo_pred',
    'math_red',
    'math_blue',
    'months_since_red',
    'months_since_blue',
    'months_since_diff',
    'win_streak_red',
    'lose_streak_red',
    'win_streak_blue',
    'lose_streak_blue',
    'win_pct_red',
    'win_pct_blue',
    'num_fights_red',
    'num_fights_blue',
    'num_wins_red',
    'num_wins_blue',
    'num_losses_red',
    'num_losses_blue',
    'num_fights_diff',
    'win_streak_diff',
    'lose_streak_diff',
    'wins_diff',
    'losses_diff',
    'win_pct_diff',
    'decision_wins_red',
    'ko_wins_red',
    'sub_wins_red',
    'decision_wins_blue',
    'ko_wins_blue',
    'sub_wins_blue',
    'decision_wins_diff',
    'ko_wins_diff',
    'sub_wins_diff',
    'decision_losses_red',
    'ko_losses_red',
    'sub_losses_red',
    'decision_losses_blue',
    'ko_losses_blue',
    'sub_losses_blue',
    'decision_losses_diff',
    'ko_losses_diff',
    'sub_losses_diff',
    'ko_pct_red',
    'dec_pct_red',
    'sub_pct_red',
    'ko_pct_blue',
    'dec_pct_blue',
    'sub_pct_blue',
    'ko_pct_diff',
    'dec_pct_diff',
    'sub_pct_diff',
    'womens_fight',
    'red_title_fights',
    'blue_title_fights',
    'red_title_wins',
    'blue_title_wins',
    'red_title_losses',
    'blue_title_losses',
    'red_title_win_pct',
    'blue_title_win_pct',
    'fav_counts_red',
    'dog_counts_red',
    'fav_counts_blue',
    'dog_counts_blue',
    'fav_counts_diff',
    'dog_counts_diff',
    'proba_fair_open_diff',

    'td_defended_pm_diff',
    'td_defended_total_diff',
    'td_total_attempted_against_diff',
    'td_total_landed_against_diff',
    'sig_str_total_attempted_against_diff',
    'sig_str_total_landed_against_diff',
    'opp_age_diff',
    'opp_reach_diff',
    'adjusted_sig_str_diff',
    'adjusted_control_diff',
    'adjusted_td_diff',
]


In [ ]:
VAL_START_DATE = pd.to_datetime(VAL_START_DATE)
TEST_START_DATE = pd.to_datetime(TEST_START_DATE)



fp = settings.all_feats_dir / 'entire_odds_stats_2026-10-11.csv'
df_model = pd.read_csv(fp)
df_model['date'] = pd.to_datetime(df_model['date'], errors='raise').dt.normalize()

categorical_cols = ['math_red', 'math_blue', 'elo_pred', 'womens_fight']
numerical_cols = [col for col in feats_to_test if col not in categorical_cols]
# Fail on unexpected strings instead of silently omitting numerical inputs.
df_model[numerical_cols] = df_model[numerical_cols].apply(pd.to_numeric, errors='raise')
df_model[categorical_cols] = df_model[categorical_cols].astype('category')

# Training starts after the configured year/month/day (exclusive lower bound).
builder = TrainTestBuilder(
    df=df_model, feats=feats_to_test, target_col='winner', date_col='date',
    odds_type='open', year=TRAIN_START_YEAR, month=TRAIN_START_MONTH,
    day=TRAIN_START_DAY, required_features_only=True,
)

# Use the same train / validation / test boundaries as model training.
pkt = builder.prepare_train_test(
    val_start_date=VAL_START_DATE, test_start_date=TEST_START_DATE,
    scale=True, save_split=False,
)
X_train, y_train = pkt['X_train'], pkt['y_train']
X_val, y_val = pkt['X_val'], pkt['y_val']
X_test, y_test = pkt['X_test'], pkt['y_test']
train_df, val_df, test_df = pkt['train_df'], pkt['val_df'], pkt['test_df']
scaler, encoder = pkt['scaler'], pkt['encoder']

assert train_df['date'].lt(VAL_START_DATE).all()
assert val_df['date'].ge(VAL_START_DATE).all()
assert val_df['date'].lt(TEST_START_DATE).all()
assert test_df['date'].ge(TEST_START_DATE).all()
assert set(pkt['cat_cols']) == set(categorical_cols)
assert set(pkt['num_cols']) == set(numerical_cols)
expected_columns = list(scaler.feature_names_in_) + list(encoder.get_feature_names_out())
assert list(X_train.columns) == list(X_val.columns) == expected_columns
assert X_train.index.equals(y_train.index) and X_val.index.equals(y_val.index)
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_val.to_numpy(dtype=float)).all()
assert 'split_paths' not in pkt

print(f'Train: {X_train.shape}; validation: {X_val.shape}')
print(f'Numerical: {len(numerical_cols)}; categorical: {len(categorical_cols)}')
print(f'Training dates: {train_df.date.min().date()} to {train_df.date.max().date()}')
print(f'Validation starts: {val_df.date.min().date()}')


In [ ]:
# Edit these paths to choose where each output is saved.
output_dir = settings.feature_selection_dir
path_plot_path = output_dir / "lasso_paths.png"
importance_plot_path = output_dir / "lasso_importance.png"
ranked_features_path = output_dir / "ranked_features.csv"
coefficient_paths_path = output_dir / "coefficient_paths.csv"

C_min, C_max, num_C = 1e-4, 1e2, 100
C_values = np.logspace(np.log10(C_max), np.log10(C_min), num_C)
coef_df, path_fig = plot_lasso_path(
    X_train, y_train, C_min=C_min, C_max=C_max, num_C=num_C,
    figsize=(14, 8), return_figure=True,
)
rank_df = rank_features_by_lasso_path(coef_df, C_values)
importance_fig = plot_lasso_importance(rank_df, top_n=30)
saved_paths = save_lasso_outputs(
    path_fig, importance_fig, rank_df, coef_df,
    path_plot_path=path_plot_path, importance_plot_path=importance_plot_path,
    ranked_features_path=ranked_features_path,
    coefficient_paths_path=coefficient_paths_path,
)
saved_paths
